Scenario 

I'm building a model to predict the optimal selling price for furniture items based on features 
like material, size, and age. 

The true market value that maximizes expected sales is y. 
In this market, the consequences of pricing mistakes are heavily skewed
Overpricing: 
    Setting the price higher than the optimal market value causes items to sit in the warehouse unsold, incurring storage costs, tying up capital, and risking inventory depreciation. 
Underpricing: 
    Setting the price below market value means the item sells quickly, but you leave a small amount of profit on the table. 

Because overpricing is far more damaging to the business, the optimization algorithm must learn to err on the side of caution—biasing predictions downward toward underpricing rather than risking an overprice.

Loss    = 5 * (y_pred - y) if y_pred > y

Loss    = 1 * (y - y_pred) if y_pred <= y

In [1259]:
import numpy as np

X = np.array([
    [1.0, 3],
    [2.0, 2],
    [3.0, 8],
    [4.0, 3]
])

y = np.array([10.0, 20.0, 30.0, 40.0])

In [ ]:
def loss_fn(y_pred, y):
    if len(y_pred) != len(y):
        raise ValueError(f"Wrong input shapes: {len(y_pred)} vs {len(y)}")
    
    loss = 0.0
    for i in range(len(y)):
        if y_pred[i] >= y[i]:
            loss += 5.0 * (y_pred[i] - y[i])
        else:
            loss += (y[i] - y_pred[i])
    return loss / len(y)

In [ ]:
n, l = X.shape
weights = np.zeros(l)
bias = 0.0

step_size = 0.05
min_step_size = 0.001
best_loss = loss_fn(X @ weights + bias, y)

In [ ]:
max_epochs = 1000

for epoch in range(max_epochs):
    epoch_improved = False
    
    for j in range(l):
        weights[j] += step_size
        loss_nudge = loss_fn(X @ weights + bias, y)
        if loss_nudge < best_loss:
            best_loss = loss_nudge
            epoch_improved = True
        else:
            weights[j] -= (2 * step_size)
            loss_nudge = loss_fn(X @ weights + bias, y)
            if loss_nudge < best_loss:
                best_loss = loss_nudge
                epoch_improved = True
            else:
                weights[j] += step_size
                
    bias += step_size
    loss_nudge = loss_fn(X @ weights + bias, y)
    if loss_nudge < best_loss:
        best_loss = loss_nudge
        epoch_improved = True
    else:
        bias -= (2 * step_size)
        loss_nudge = loss_fn(X @ weights + bias, y)
        if loss_nudge < best_loss:
            best_loss = loss_nudge
            epoch_improved = True
        else:
            bias += step_size

    if not epoch_improved:
        step_size /= 2.0
        print(f"Epoch {epoch + 1}: No progress. Decayed step_size to {step_size:.6f}")
        
        if step_size < min_step_size:
            print(f"Stopping condition reached: step_size ({step_size:.6f}) < {min_step_size}")
            break

In [ ]:
y_pred_final = X @ weights + bias
errors = y_pred_final - y

print("Weights:    ", np.round(weights, 4))
print("Bias:       ", np.round(bias, 4))
print("True Values (y):    ", y)
print("Predictions (y_pred):", np.round(y_pred_final, 2))
print("Errors:", np.round(errors, 2))